This notebook works through [an issue](https://github.com/skybristol/indian_country_data/issues/17) where Wikidata has some items that conflate Alaska Native Tribes with items representing Alaska municipalities (some of which I introduced myself in initial experimentation). It uses a scripting design pattern I've used in many other cases working against other Wikibase instances with the wikibase.cloud project. It is essentially a script that does some work but in notebook form so that I can provide a record of thinking and process as a reference. This follows a bunch of work I did to hash through existing Wikidata items and Wikipedia articles toward a baseline of identifiers and data/information content for all 575 federally recognized Native America tribes in the U.S.

In [1]:
import pandas as pd
from wbmaker import WB
import os

import gspread
from google.oauth2.service_account import Credentials

wd = WB()

# Google service account authentication
scopes = [
    'https://www.googleapis.com/auth/spreadsheets',
    'https://www.googleapis.com/auth/drive'
]
g_creds = Credentials.from_service_account_file('credentials.json', scopes=scopes)
g_client = gspread.authorize(g_creds)

# Google Sheets Source
To establish an intitial baseline of entities in Wikidata representing the officially recognized Tribes in the U.S., I'm working up some basic details in simple spreadsheet form in Google Sheets, with some back and forth reconcilliation work in OpenRefine. This lets me see things a bit more simply and share the tables as needed.

For this exercise, I'm pulling my primary "tribes" source along with a separate sheet where I store some assembled details on Alaska cities and census-designated places.

In [2]:
tribes_sheet = g_client.open('Federally Recognized Tribes').worksheet('tribes').get_all_records()
tribes = pd.DataFrame(tribes_sheet)
tribes = tribes.replace('', None)

ak_city_classes = g_client.open('Federally Recognized Tribes').worksheet('ak_city_classifications').get_all_records()
df_ak_city_classifications = pd.DataFrame(ak_city_classes)

df_conflated_ak_tribes = tribes[tribes['qid'] == tribes['qid_ak_city']].copy()
df_conflated_ak_tribes = pd.merge(
    left=df_conflated_ak_tribes,
    right=df_ak_city_classifications,
    on='qid_ak_city',
    how='left'
)

df_conflated_ak_tribes

,region,fr_label,fr_alternate_labels,old_name,qid,wikipedia_en,qid_ak_city,ak_city_type,qid_anrc,nill_index,ak_city_label,ak_city_class
0,Alaska,Alatna Village,None,None,Q1837416,"Alatna,_Alaska",Q1837416,CDP,Q5303808,alatna.html,Alatna Village,census-designated place
1,Alaska,Beaver Village,None,None,Q2313950,"Beaver,_Alaska",Q2313950,CDP,Q5303808,beaver_village.html,Beaver Village,census-designated place
2,Alaska,Birch Creek Tribe,None,Birch Creek Village,Q1837588,"Birch_Creek,_Alaska",Q1837588,None,Q5303808,birch_creek.html,Birch Creek,census-designated place
3,Alaska,Emmonak Village,None,None,Q79553,"Emmonak,_Alaska",Q79553,None,Q5021401,emmonak.html,Emmonak,second class city
4,Alaska,Holy Cross Tribe,None,None,Q79507,"Holy_Cross,_Alaska",Q79507,None,Q5303808,holy_cross.html,Holy Cross,second class city
5,Alaska,"Metlakatla Indian Community, Annette Island Re...",None,None,Q2037538,"Metlakatla,_Alaska",Q2037538,None,Q1666068,metlakatla.html,Metlakatla,census-designated place
6,Alaska,Native Village of Atqasuk,None,None,Q79714,"Atqasuk,_Alaska",Q79714,None,Q4787552,atqasuk.html,Atqasuk,second class city
7,Alaska,Native Village of Gambell,None,None,Q79677,"Gambell,_Alaska",Q79677,None,Q4891841,native_gambell.html,Gambell,second class city
8,Alaska,Native Village of Venetie Tribal Government,Arctic Village|Village of Venetie|Artic Village,None,Q638197,Native_Village_of_Venetie_Tribal_Government,Q638197,None,Q5303808,native_venetie.html,Arctic Village,census-designated place
9,Alaska,Northway Village,None,None,Q2090657,"Northway_Village,_Alaska",Q2090657,None,Q5303808,northway.html,Northway Village,census-designated place


# Reference Values/Objects
This stuff ultimately needs to move to some type of utility system for working with Wikidata. In past work on wikibase.cloud, I had something (built into my WBMaker Python package) that read out property and classification content into useful reference structures. That's impractical at the scale of Wikidata, so I'm contemplating something that would use an encoding of one or more specific data models that I want to operate against.

In [3]:
q = {
    'US tribe': 'Q7840353',
    '89 FR 944': 'Q127419548', # 2024 list of indian entities
    'United States': 'Q30',
    'Alaska': 'Q797',
    'English': 'Q1860'
}

p = {
    'instance of': 'P31',
    'described at URL': 'P973',
    'member count': 'P2124',
    'native label': 'P1705',
    'different from': 'P1889',
    'part of': 'P361',
    'stated in': 'P248',
    'reference URL': 'P854',
    'country': 'P17',
    'located in the administrative territorial entity': 'P131',
    'language of work or name': 'P407',
    'headquarters location': 'P159'
}

fr_notice_ref = wd.datatypes.Item(
    value=q['89 FR 944'], 
    prop_nr=p['stated in']
)
fed_tribe_instance_claim = wd.datatypes.Item(
    value=q['US tribe'], 
    prop_nr=p['instance of'], 
    references=[fr_notice_ref]
)

us_country_claim = wd.datatypes.Item(
    value=q['United States'], 
    prop_nr=p['country']
)

ak_loc_claim = wd.datatypes.Item(
    value=q['Alaska'], 
    prop_nr=p['located in the administrative territorial entity']
)

en_lang_qualifier = wd.datatypes.Item(
    value=q['English'], 
    prop_nr=p['language of work or name']
)

# Run workflow
This is a one-off process, so a simple workflow script makes enough sense here. I'm doing the following:
* Pull up the city item to get stuff from it and remove stuff specific to the tribe
* Remove the tribe label as an alias (I'll run back through a quick review to make sure there isn't anything else tribe-specific in the labels)
* Create a new tribe item with the basics (instance of, location)
* Remove the fed tribe classification from the city and change its description
* Add part of relationship to ANRC
* Add described at URL to the NILL Tribal Law Gateway index page for the tribe
* Add different from claims to both the tribe and city (requires another pass on the city item)

I'm printing out the new identifiers as URLs so I can go in and handle a few things manually that are easier that way:
* Move official website from city to tribe if appropriate
* Tweak anything else out of place
* Review what the city items have in site links and if these need to shift or go onto a list for future Wikipedia work

Note: I ran into issues the way I first had this laid out in that a bunch of the city items in Wikidata have some problem in their claims that is preventing me from being able to write changes on them to Wikidata. I've dealt with this before where something like lack of a precision value on a property that requires it causes an error to be thrown. Rather than worry about that right now, I opted to simply create the new tribe items and go fix the city items manually. I'll try to suss out where the issues are and fix them if I can.

In [4]:
new_tribe_items = []

for _, row in df_conflated_ak_tribes.iterrows():
    # Get the city item we need to modify
    city_item = wd.wbi.item.get(row['qid'])

    # Create new tribe item
    tribe_item = wd.wbi.item.new()

    # Set labels and descriptions for tribe item
    tribe_item.labels.set('en', row['fr_label'].strip())
    tribe_item.descriptions.set('en', 'federally recognized tribe in Alaska, United States')

    # Set standardized claims for AK tribes
    tribe_item.claims.add(fed_tribe_instance_claim)
    tribe_item.claims.add(us_country_claim)
    tribe_item.claims.add(ak_loc_claim)

    # Transfer member count claims from city to tribe
    member_count_claims = city_item.claims.get(p['member count'])
    if member_count_claims:
        for c in member_count_claims:
            c.id = None
            tribe_item.claims.add(c)

    # Add native label claim if present
    native_label_claims = city_item.claims.get(p['native label'])
    if native_label_claims:
        for c in native_label_claims:
            c.id = None
            tribe_item.claims.add(c)

    # Add headquarters location claim if present
    hq_location_claims = city_item.claims.get(p['headquarters location'])
    if hq_location_claims:
        for c in hq_location_claims:
            c.id = None
            tribe_item.claims.add(c)

    # Add part of claim for ANRC relationship
    part_of_claim = wd.datatypes.Item(
        prop_nr=p['part of'],
        value=row['qid_anrc']
    )
    tribe_item.claims.add(part_of_claim)

    # Add described at URL claim for NILL index page
    described_at_claim = wd.datatypes.URL(
        prop_nr=p['described at URL'],
        value=f"https://narf.org/nill/tribes/{row['nill_index']}",
        qualifiers=[en_lang_qualifier]
    )
    tribe_item.claims.add(described_at_claim)

    # Add different from claim on tribe
    claim_diff = wd.datatypes.Item(
        prop_nr=p['different from'],
        value=row['qid_ak_city']
    )
    tribe_item.claims.add(claim_diff)

    # Write the new tribe item
    tribe_response = tribe_item.write(summary='Created new item for AK Native Tribe to fix conflation with associated AK city')
    print(f'Created tribe item ({row["fr_label"].strip()}): https://www.wikidata.org/entity/{tribe_response.id}')
    print(f'Go work on city item: https://www.wikidata.org/entity/{row["qid_ak_city"]}')
    
    # Add identifiers/names to a list for future reference
    new_tribe_items.append({
        'tribe_name': row['fr_label'],
        'tribe_qid': tribe_response.id
    })

Created tribe item (Alatna Village): https://www.wikidata.org/entity/Q137998427
Go work on city item: https://www.wikidata.org/entity/Q1837416
Created tribe item (Beaver Village): https://www.wikidata.org/entity/Q137998428
Go work on city item: https://www.wikidata.org/entity/Q2313950
Created tribe item (Birch Creek Tribe): https://www.wikidata.org/entity/Q137998429
Go work on city item: https://www.wikidata.org/entity/Q1837588
Created tribe item (Emmonak Village): https://www.wikidata.org/entity/Q137998430
Go work on city item: https://www.wikidata.org/entity/Q79553
Created tribe item (Holy Cross Tribe): https://www.wikidata.org/entity/Q137998431
Go work on city item: https://www.wikidata.org/entity/Q79507
Created tribe item (Metlakatla Indian Community, Annette Island Reserve): https://www.wikidata.org/entity/Q137998432
Go work on city item: https://www.wikidata.org/entity/Q2037538
Created tribe item (Native Village of Atqasuk): https://www.wikidata.org/entity/Q137998433
Go work on c